<a href="https://colab.research.google.com/github/AmitoVrito/Traceprop/blob/main/notebooks/exp25_llm_inline_overhead_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Traceprop-LLM — inline gradient provenance (MLSys)

**Claim:** capturing per-sample gradient provenance *inline* during a LoRA fine-tune costs **<1% wall-clock**, vs the full extra pass over the training set that post-hoc methods (TRAK / LoGRA) require.

Run on a GPU runtime (Runtime → Change runtime type → GPU; T4/L4 fine, A100 for the largest).

Sections:
1–5. **Overhead** — baseline vs inline-instrumented step time on GPT-2 and Pythia-410M/1B (`exp25`); tracked-blocks tradeoff; results table.
6. **Head-to-head** — inline logging vs a post-hoc extraction sweep (`exp26`): the ~30×/~150× cost win that is the paper's centerpiece.

## 1. Install dependencies

In [1]:
!pip -q install transformers peft accelerate
# Colab ships an old torchao (0.10) whose PEFT compatibility check raises on import; we don't use it.
!pip -q uninstall -y torchao 2>/dev/null
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

torch 2.11.0+cu128 cuda True NVIDIA L4


## 2. Get the Traceprop code

The `traceprop.llm` module and `exp25` are on the private repo. Paste a GitHub token (a fine-grained read token for `AmitoVrito/Traceprop`) below. If you prefer, upload the repo zip instead and skip this cell.

In [2]:
import getpass, os
TOKEN = getpass.getpass('GitHub token (leave blank if uploading manually): ').strip()
if TOKEN:
    url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
    !git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
    %cd /content/Traceprop
    !pip -q install -e .
else:
    print('No token given — upload the repo to /content/Traceprop, then run: %cd /content/Traceprop and !pip install -e .')

GitHub token (leave blank if uploading manually): ··········
/content/Traceprop
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for traceprop (pyproject.toml) ... done


In [3]:
# sanity: import the inline logger
import sys; sys.path.insert(0, '/content/Traceprop/experiments')
from traceprop.llm import LoRAGradientLogger, select_lora_linears
print('traceprop.llm OK')

traceprop.llm OK


## 3. Headline: overhead on GPT-2 (124M) + LoRA

Baseline = plain LoRA step. Instrumented = same step + inline per-sample gradient logging. The store footprint is what you keep for attribution.

Each run reports **two** overhead numbers:
- **`synced%`** — conservative: a GPU sync is forced every step, so the projection is fully serialized against the training step. Upper bound.
- **`throughput%`** — realistic: projected gradients are buffered on-device and the run is synced once, so the projection overlaps with compute. This is the actual cost of turning logging on in a training loop, and the headline number.

`--track 1` logs only the **last transformer block** (last-layer attribution, i.e. Traceprop-LL): the projected-gradient dimension — and thus the projection cost — scales with tracked parameters, so last-block is the cheap regime. Batch/seq are held **fixed across all three models** so the only variable is model size.

In [ ]:
%cd /content/Traceprop/experiments
!python exp25_llm_inline_overhead.py --backend hf --model gpt2 --device cuda \
    --steps 120 --warmup 15 --repeats 20 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1

## 4. Scale up: Pythia-410M (and 1B on A100)

In [ ]:
!python exp25_llm_inline_overhead.py --backend hf --model EleutherAI/pythia-410m --device cuda \
    --steps 120 --warmup 15 --repeats 20 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1

In [ ]:
# Pythia-1B — fits on L4/A100 (24GB+). Same batch/seq as the others for a fair size comparison.
!python exp25_llm_inline_overhead.py --backend hf --model EleutherAI/pythia-1b --device cuda \
    --steps 120 --warmup 15 --repeats 20 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1

## 4b. Tradeoff: overhead vs number of tracked blocks (GPT-2)

Shows the knob directly — overhead grows with how many blocks you log. Last-block (`--track 1`) is the sub-1% attribution regime; full-model (`--track 0`) is the expensive end. Quality-vs-overhead for these settings is workstream C (LDS parity).

In [ ]:
import sys; sys.path.insert(0, '/content/Traceprop/experiments')
from types import SimpleNamespace
from exp25_llm_inline_overhead import run

sweep = []
for t in [1, 2, 6, 0]:   # last-1, last-2, last-6 blocks, then all layers
    ns = SimpleNamespace(backend='hf', model='gpt2', device='cuda', steps=100, warmup=15,
                         repeats=20, batch=8, seq=128, rank=8, proj_dim=512, d=256, n_blocks=2,
                         track=t, factored=False, kfac=16, dtype='fp32')
    r = run(ns)
    sweep.append((t, r['n_tracked_layers'], r['per_sample_grad_dim'],
                  r['overhead_pct'], r['overhead_std']))

print('\n=== overhead vs tracked blocks (gpt2, batch8 seq128, synced median±std) ===')
print(f"{'track':>6}{'layers':>8}{'grad_dim':>12}{'synced%':>12}")
for t, l, g, o, sd in sweep:
    label = 'all' if t == 0 else f'last-{t}'
    print(f"{label:>6}{l:>8}{g:>12}{o:>8.2f}±{sd:<4.2f}")

## 5. Collect results (per-model overhead table)

In [ ]:
import glob, json
rows = [json.load(open(p)) for p in glob.glob('/content/Traceprop/experiments/results/exp25_hf_*track1.json')]
print(f"{'model':<22}{'base_ms':>9}{'synced%':>14}{'throughput%':>16}{'store_mb':>10}")
for r in sorted(rows, key=lambda r: r['model']):
    syn = f"{r['overhead_pct']:.2f}±{r['overhead_std']:.2f}"
    thr = f"{r['throughput_overhead_pct']:.2f}±{r['throughput_overhead_std']:.2f}"
    print(f"{r['model']:<22}{r['base_step_ms']:>9.2f}{syn:>14}{thr:>16}{r['store_mb']:>10.2f}")
print('\nsynced%     = conservative (projection serialized against each step)')
print('throughput% = realistic training cost (projection overlaps compute)')

## 6. Head-to-head: inline logging vs post-hoc extraction (the centerpiece)

`exp26` measures, on the same LoRA model and a fixed N-sample training set, the cost to produce the per-sample gradient store two ways:

- **Post-hoc** (TRAK / LoGRA): a dedicated forward+backward sweep over the whole training set *after* training. TRAK ensembles over K checkpoints → K sweeps.
- **Inline** (Traceprop): the marginal cost folded into the training pass you already run.

`speedup = post-hoc extraction wall-clock / inline marginal wall-clock`. This is what turns "~3% overhead" into "~30× (single ckpt) / ~150× (TRAK 5-ckpt) cheaper to reach attribution-ready."

In [ ]:
%cd /content/Traceprop/experiments
!python exp26_posthoc_vs_inline.py --backend hf --model gpt2 --device cuda \
    --n_samples 512 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1 --repeats 20 --trak_ckpts 5

In [ ]:
!python exp26_posthoc_vs_inline.py --backend hf --model EleutherAI/pythia-410m --device cuda \
    --n_samples 512 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1 --repeats 20 --trak_ckpts 5

In [ ]:
!python exp26_posthoc_vs_inline.py --backend hf --model EleutherAI/pythia-1b --device cuda \
    --n_samples 384 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1 --repeats 20 --trak_ckpts 5

In [ ]:
import glob, json
rows = [json.load(open(p)) for p in glob.glob('/content/Traceprop/experiments/results/exp26_hf_*track1.json')]
print(f"{'model':<22}{'flush_s':>10}{'overhead%':>11}{'posthoc_s':>11}{'LoGRA x':>9}{'TRAK x':>9}")
for r in sorted(rows, key=lambda r: r['model']):
    trak = [v for k,v in r.items() if k.startswith('speedup_vs_trak')][0]
    print(f"{r['model']:<22}{r['inline_flush_s']:>10.3f}{r['inline_flush_overhead_pct']:>11.2f}"
          f"{r['posthoc_pass_s']:>11.2f}{r['speedup_vs_logra_1ckpt']:>9.1f}{trak:>9.1f}")
print("\nflush_s   = directly-timed (synced) inline logging cost per pass — the reliable number")
print("posthoc_s = dedicated extraction sweep (TRAK/LoGRA);  speedup = posthoc / flush (x K for TRAK)")
print("(inline_marginal_s in the JSON is the pass-difference — noisy at sub-1%; use flush_s.)")